# Notebook 03b — Modelos v2 (Features Verdes + Mesenquimal)

**Target**: `log1p(Tumor_Telomere fusion rate)` — transformación log para corregir skewness=49  
**Features (primera aproximación — verdes)**:
- `firstResponse` (PR / PD / SD / CR / Sin_dato)
- `treatment_agrupado` (top 15 tratamientos + Otro + Sin_dato)
- `Mesenchimal_origin` (Mesenchimal / Non_mesenchimal / Mix)

**Modelos**: Lasso → Ridge → Elastic Net  
**NaN strategy**: categoría `Sin_dato` — conservamos todos los 4535 pacientes  
**Join mesenquimal**: por `cancer_type_code` (los nombres completos no coinciden con los códigos del archivo)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import scipy.stats as stats

import os
os.chdir('..')  

print('Librerías cargadas OK')

## 1. Carga y unión de datos

In [ ]:
# Base: pacientes limpios
df = pd.read_csv('datos_limpios_modelo_a.csv')
print(f'Dataset base: {df.shape}')

# Features verdes: firstResponse y treatment (de revision_color_code)
df_full = pd.read_excel('revision_color_code.xlsx', sheet_name='Sheet1',
                        usecols=['Patient_ID_IF_mod', 'firstResponse', 'treatment'])
df = df.merge(df_full, on='Patient_ID_IF_mod', how='left')
print(f'Tras unir firstResponse y treatment: {df.shape}')

# Variable mesenquimal: join por cancer_type_CODE (no por nombre completo)
df_mes = pd.read_excel('Tumor_mesenchimal.xlsx')
df_mes.columns = ['cancer_type_code', 'Mesenchimal_origin']
df = df.merge(df_mes, on='cancer_type_code', how='left')
print(f'Tras unir mesenquimal: {df.shape}')
print()
print('Distribución Mesenchimal_origin:')
print(df['Mesenchimal_origin'].value_counts(dropna=False))

## 2. Target — distribución y transformación log1p

In [ ]:
TARGET = 'Tumor_Telomere fusion rate (Gt-end/coverage*purity)'
y_raw = df[TARGET].copy()

print('=== Target original ===')
print(y_raw.describe())
print(f'Skewness: {y_raw.skew():.2f}')
print(f'% ceros exactos: {(y_raw == 0).mean()*100:.1f}%')

# Transformación log1p para corregir la asimetría extrema
y = np.log1p(y_raw)
print(f'\nSkewness tras log1p: {y.skew():.2f}')

## 3. Preparación de features

In [ ]:
# NaN → 'Sin_dato' (mantenemos todos los pacientes)
df['firstResponse'] = df['firstResponse'].fillna('Sin_dato')
df['Mesenchimal_origin'] = df['Mesenchimal_origin'].fillna('Sin_dato')

# Treatment: 575 valores únicos → top 15 + 'Otro'
top15 = df['treatment'].value_counts().head(15).index.tolist()
df['treatment_agrupado'] = df['treatment'].apply(
    lambda x: x if x in top15 else ('Sin_dato' if pd.isna(x) else 'Otro')
)

print('firstResponse:')
print(df['firstResponse'].value_counts())
print()
print('treatment_agrupado:')
print(df['treatment_agrupado'].value_counts())
print()
print('Mesenchimal_origin:')
print(df['Mesenchimal_origin'].value_counts())

In [ ]:
# Dummies de las 3 variables categóricas
X = pd.get_dummies(
    df[['firstResponse', 'treatment_agrupado', 'Mesenchimal_origin']],
    drop_first=True
)
print(f'Shape X: {X.shape}  |  Shape y: {y.shape}')
print(f'Features: {X.columns.tolist()}')

## 4. Correlación features vs target

In [ ]:
print('Top 10 correlaciones Pearson (features vs target log1p):')
cors = {col: stats.pearsonr(X[col], y) for col in X.columns}
top = sorted(cors.items(), key=lambda x: abs(x[1][0]), reverse=True)[:10]
for col, (r, p) in top:
    print(f'  {col:45s}  r={r:+.3f}  p={p:.4f}')

## 5. Split train / test + escalado

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

## 6. Función de evaluación

In [ ]:
def evaluar_modelo(nombre, modelo, X_tr, X_te, y_tr, y_te, feature_names):
    modelo.fit(X_tr, y_tr)
    y_pred_log  = modelo.predict(X_te)
    # Deshacer log1p → métricas en espacio original
    y_pred_orig = np.expm1(y_pred_log)
    y_te_orig   = np.expm1(np.array(y_te))

    r2   = r2_score(y_te_orig, y_pred_orig)
    mae  = mean_absolute_error(y_te_orig, y_pred_orig)
    rmse = np.sqrt(mean_squared_error(y_te_orig, y_pred_orig))
    cv   = cross_val_score(modelo, X_tr, y_tr, cv=5, scoring='r2').mean()

    sep = '=' * 50
    print(sep)
    print(f'  {nombre}')
    print(sep)
    print(f'  R²  (test, espacio original):  {r2:.4f}')
    print(f'  R²  (CV-5, espacio log):       {cv:.4f}')
    print(f'  MAE  (espacio original):        {mae:.4f}')
    print(f'  RMSE (espacio original):        {rmse:.4f}')

    coefs = pd.Series(modelo.coef_, index=feature_names)
    nz    = coefs[coefs != 0].sort_values(key=abs, ascending=False)
    print(f'  Coefs no nulos: {len(nz)} / {len(coefs)}')
    if len(nz) > 0:
        print(nz.head(10).to_string())

    return {'nombre': nombre, 'modelo': modelo, 'R2_test': r2, 'R2_cv': cv,
            'MAE': mae, 'RMSE': rmse, 'y_pred': y_pred_orig}

## 7. Modelo 1 — Lasso

In [ ]:
lasso = Lasso(alpha=0.001, max_iter=10000, random_state=42)
res_lasso = evaluar_modelo('Lasso (alpha=0.001)', lasso,
                            X_train_sc, X_test_sc, y_train, y_test,
                            X.columns)

## 8. Modelo 2 — Ridge

In [ ]:
ridge = Ridge(alpha=1.0, random_state=42)
res_ridge = evaluar_modelo('Ridge (alpha=1.0)', ridge,
                            X_train_sc, X_test_sc, y_train, y_test,
                            X.columns)

## 9. Modelo 3 — Elastic Net

In [ ]:
enet = ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=42)
res_enet = evaluar_modelo('Elastic Net (alpha=0.001, l1_ratio=0.5)', enet,
                           X_train_sc, X_test_sc, y_train, y_test,
                           X.columns)

## 10. Comparativa final

In [ ]:
resultados = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('modelo', 'y_pred')}
    for r in [res_lasso, res_ridge, res_enet]
]).set_index('nombre')

print('=== COMPARATIVA FINAL ===')
print(resultados.round(4).to_string())
print()
mejor = resultados['R2_test'].idxmax()
print(f'Mejor modelo (R² test): {mejor}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, res in zip(axes, [res_lasso, res_ridge, res_enet]):
    y_pred = np.array(res['y_pred'])
    y_real = np.expm1(np.array(y_test))   # deshacer log1p
    ax.scatter(y_real, y_pred, alpha=0.3, s=10, color='steelblue')
    lim = max(y_real.max(), y_pred.max())
    ax.plot([0, lim], [0, lim], 'r--', linewidth=1, label='Perfecto')
    ax.set_xlabel('Real (espacio original)')
    ax.set_ylabel('Predicho (espacio original)')
    ax.set_title(f"{res['nombre']}\nR²={res['R2_test']:.4f}")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('notebooks/img_comparativa_modelos_v2.png', dpi=150, bbox_inches='tight')
plt.close()
print('Guardado: notebooks/img_comparativa_modelos_v2.png')